# Remote sensing ensemble mean and exportation

In [1]:
# import the libraries
import ee
import pandas as pd
import os
import numpy as np
import random
from random import sample
import itertools 
import geopandas as gpd
from sklearn.metrics import r2_score
from termcolor import colored # this is allocate colour and fonts type for the print title and text
from IPython.display import display, HTML

In [2]:
# initialize the earth engine API
ee.Initialize()

## load all the predicted potential biomass maps into a multiband image

In [3]:
# load the images for the 30 ensemble models
# define an empty image
firstImage = ee.Image('users/leonidmoore/ForestBiomass/RemoteSensingModel/PredictedMaps/Predicted_Potential_Remote_Sensing_Biomass_Map_with_Seed_0').rename('Model_0')

modelList = np.arange(1, 100, 1).tolist()
for ml in modelList:
    perModelImage = ee.Image('users/leonidmoore/ForestBiomass/RemoteSensingModel/PredictedMaps/Predicted_Potential_Remote_Sensing_Biomass_Map_with_Seed_'+str(ml)).rename('Model_'+str(ml))
    firstImage = firstImage.addBands(perModelImage)

print(colored('The band names are:', 'blue', attrs=['bold']),firstImage.bandNames().getInfo())

The band names are: ['Model_0', 'Model_1', 'Model_2', 'Model_3', 'Model_4', 'Model_5', 'Model_6', 'Model_7', 'Model_8', 'Model_9', 'Model_10', 'Model_11', 'Model_12', 'Model_13', 'Model_14', 'Model_15', 'Model_16', 'Model_17', 'Model_18', 'Model_19', 'Model_20', 'Model_21', 'Model_22', 'Model_23', 'Model_24', 'Model_25', 'Model_26', 'Model_27', 'Model_28', 'Model_29', 'Model_30', 'Model_31', 'Model_32', 'Model_33', 'Model_34', 'Model_35', 'Model_36', 'Model_37', 'Model_38', 'Model_39', 'Model_40', 'Model_41', 'Model_42', 'Model_43', 'Model_44', 'Model_45', 'Model_46', 'Model_47', 'Model_48', 'Model_49', 'Model_50', 'Model_51', 'Model_52', 'Model_53', 'Model_54', 'Model_55', 'Model_56', 'Model_57', 'Model_58', 'Model_59', 'Model_60', 'Model_61', 'Model_62', 'Model_63', 'Model_64', 'Model_65', 'Model_66', 'Model_67', 'Model_68', 'Model_69', 'Model_70', 'Model_71', 'Model_72', 'Model_73', 'Model_74', 'Model_75', 'Model_76', 'Model_77', 'Model_78', 'Model_79', 'Model_80', 'Model_81', 'Mode

In [4]:
# define the boundary geography reference
unboundedGeo = ee.Geometry.Polygon([-180, 88, 0, 88, 180, 88, 180, -88, 0, -88, -180, -88], None, False)

## Calculate the Mean and variation coefficient maps of Potential biomass

In [5]:
# calculate the mean and variation images
meanImage = firstImage.reduce(ee.Reducer.mean())
variImage = firstImage.reduce(ee.Reducer.stdDev()).divide(meanImage)

percentileImage = firstImage.reduce(ee.Reducer.percentile([2.5,97.5],['lower','upper']))

## Export Mean and variation coefficient maps of wood density mean to Asset

In [6]:
# add those two images into the GEE assets
meanExport = ee.batch.Export.image.toAsset(image = meanImage,
                                           description = '20210614_RemoteSensing_Biomass_Mean_Ensemble_Map_To_Asset',
                                           assetId = 'users/leonidmoore/ForestBiomass/RemoteSensingModel/EnsambledMaps/Ensambled_Potential_Remote_Sensing_Biomass_Map_Mean',
                                           region = unboundedGeo,
                                           crs = 'EPSG:4326',
                                           crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                           maxPixels = 1e13)


# start the export task
meanExport.start()
# show the task status
meanExport.status()

variExport = ee.batch.Export.image.toAsset(image = variImage,
                                           description = '20210614_RemoteSensing_Biomass_Variation_Coef_Ensemble_Map_To_Asset',
                                           assetId = 'users/leonidmoore/ForestBiomass/RemoteSensingModel/EnsambledMaps/Ensambled_Potential_Remote_Sensing_Biomass_Map_Variation_Coefficient',
                                           region = unboundedGeo,
                                           crs = 'EPSG:4326',
                                           crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                           maxPixels = 1e13)

# start the export task
# variExport.start()
# show the task status
# variExport.status()

## Export the 95% condifence interval maps

In [16]:
percentileExport = ee.batch.Export.image.toAsset(image = percentileImage,
                                           description = '20210614_RemoteSensing_Biomass_Percentile_Ensemble_Map_To_Asset',
                                           assetId = 'users/leonidmoore/ForestBiomass/RemoteSensingModel/EnsambledMaps/Ensambled_Potential_Remote_Sensing_Biomass_Map_Percentile',
                                           region = unboundedGeo,
                                           crs = 'EPSG:4326',
                                           crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                           maxPixels = 1e13)

# start the export task
percentileExport.start()
# show the task status
percentileExport.status()

{'state': 'READY',
 'description': '20210614_RemoteSensing_Biomass_Percentile_Ensemble_Map_To_Asset',
 'creation_timestamp_ms': 1623763146279,
 'update_timestamp_ms': 1623763146279,
 'start_timestamp_ms': 0,
 'task_type': 'EXPORT_IMAGE',
 'id': 'JR2DVU54TCG7FKD6JD7JS525',
 'name': 'projects/earthengine-legacy/operations/JR2DVU54TCG7FKD6JD7JS525'}

## Get the ensemble mean of the direct present potential models

In [25]:
# load the images for the 100 ensemble models
# define an empty image
firstImage = ee.Image('users/leonidmoore/ForestBiomass/RemoteSensingModel/PredictedMaps/Predicted_Conservation_Potential_Remote_Sensing_Biomass_Map_with_Seed_0').rename('Model_0')

modelList = np.arange(1, 100, 1).tolist()
for ml in modelList:
    perModelImage = ee.Image('users/leonidmoore/ForestBiomass/RemoteSensingModel/PredictedMaps/Predicted_Conservation_Potential_Remote_Sensing_Biomass_Map_with_Seed_'+str(ml)).rename('Model_'+str(ml))
    firstImage = firstImage.addBands(perModelImage)

print(colored('The band names are:', 'blue', attrs=['bold']),firstImage.bandNames().getInfo())

The band names are: ['Model_0', 'Model_1', 'Model_2', 'Model_3', 'Model_4', 'Model_5', 'Model_6', 'Model_7', 'Model_8', 'Model_9', 'Model_10', 'Model_11', 'Model_12', 'Model_13', 'Model_14', 'Model_15', 'Model_16', 'Model_17', 'Model_18', 'Model_19', 'Model_20', 'Model_21', 'Model_22', 'Model_23', 'Model_24', 'Model_25', 'Model_26', 'Model_27', 'Model_28', 'Model_29', 'Model_30', 'Model_31', 'Model_32', 'Model_33', 'Model_34', 'Model_35', 'Model_36', 'Model_37', 'Model_38', 'Model_39', 'Model_40', 'Model_41', 'Model_42', 'Model_43', 'Model_44', 'Model_45', 'Model_46', 'Model_47', 'Model_48', 'Model_49', 'Model_50', 'Model_51', 'Model_52', 'Model_53', 'Model_54', 'Model_55', 'Model_56', 'Model_57', 'Model_58', 'Model_59', 'Model_60', 'Model_61', 'Model_62', 'Model_63', 'Model_64', 'Model_65', 'Model_66', 'Model_67', 'Model_68', 'Model_69', 'Model_70', 'Model_71', 'Model_72', 'Model_73', 'Model_74', 'Model_75', 'Model_76', 'Model_77', 'Model_78', 'Model_79', 'Model_80', 'Model_81', 'Mode

In [26]:
# calculate the mean and variation images
meanPoImage = firstImage.reduce(ee.Reducer.mean())

In [27]:
presentCOnservationPoExport = ee.batch.Export.image.toAsset(image = meanPoImage,
                                           description = '20210625_RemoteSensing_Biomass_Present_Conservation_Potential_Ensemble_Map_To_Asset',
                                           assetId = 'users/leonidmoore/ForestBiomass/RemoteSensingModel/EnsambledMaps/Remote_Sensing_Ensambled_Present_Conservation_Potential_Biomass_Map',
                                           region = unboundedGeo,
                                           crs = 'EPSG:4326',
                                           crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                           maxPixels = 1e13)

# start the export task
presentCOnservationPoExport.start()
# show the task status
presentCOnservationPoExport.status()

{'state': 'READY',
 'description': '20210625_RemoteSensing_Biomass_Present_Conservation_Potential_Ensemble_Map_To_Asset',
 'creation_timestamp_ms': 1624622884985,
 'update_timestamp_ms': 1624622884985,
 'start_timestamp_ms': 0,
 'task_type': 'EXPORT_IMAGE',
 'id': 'CHBBTS2JXKZA7BFBUL2QCN2I',
 'name': 'projects/earthengine-legacy/operations/CHBBTS2JXKZA7BFBUL2QCN2I'}